# Deploy Identity Provider (Cognito or Okta)

This tutorial runs end-to-end on **either Amazon Cognito or Okta**, selected by a single top-level flag `IDP_PROVIDER`. You choose it **once** in this notebook (Step 0 cell); it persists to SSM and every downstream notebook + deploy script reads it back. Only the path matching your flag runs below.

## Prerequisites

- ✅ AWS credentials configured (via AWS CLI, environment variables, or `.env` file)
- ✅ Python 3.10+ with virtual environment and dependencies installed

**Per identity provider:**

| Provider | Prerequisites |
|---|---|
| `cognito` (default) | An AWS account — the tutorial creates the Cognito user pool for you. |
| `okta` | An AWS account **plus** a free [Okta Developer](https://developer.okta.com/) tenant. Set `OKTA_ORG_URL` and `OKTA_API_TOKEN` in your `.env` (the Okta setup reads them). |

## What This Notebook Does

1. Chooses + persists `IDP_PROVIDER` (Step 0)
2. Runs the matching IdP setup (Step 1): Cognito user pool / Okta OIDC + OBO apps, groups, test users
3. Verifies configuration in SSM (Step 2)

## Next Notebook

- **02-deploy-iam-roles.ipynb**

In [ ]:
%pip install -r requirements.txt

In [ ]:
# AWS Initialization - Load credentials and create session (IdP-agnostic)
from utils.notebook_init import init_aws

# This will:
# 1. Load credentials from .env file (if it exists)
# 2. Create and validate AWS session (env vars take precedence over SSO)
# 3. Return session, region, and account_id for use in this notebook
session, region, account_id = init_aws()

# SSM is the cross-notebook coordination substrate (used on both IdP paths)
ssm_client = session.client("ssm", region_name=region)

print("✅ Ready to proceed with AWS operations")
print(f"   Account ID: {account_id}")
print(f"   Region: {region}")

## Step 0: Choose & Persist the Identity Provider

You choose the IdP **here**, by editing the Step-0 cell below (`cognito` default, or `okta`). `set_idp_provider` validates the value and writes it to SSM (`/app/lakehouse-agent/idp-provider`); every downstream notebook and deploy script reads it back with `get_idp_provider`. Default is `cognito`, so an unmodified checkout reproduces the standard Cognito tutorial. (The Okta path also needs `OKTA_ORG_URL` + `OKTA_API_TOKEN` in `.env`.)

In [ ]:
from utils.idp_config import set_idp_provider

# ── CHOOSE YOUR IDENTITY PROVIDER HERE: "cognito" (default) or "okta" ──
# This cell is the ONE place you choose the IdP — edit the value below.
# It's validated + persisted to SSM (/app/lakehouse-agent/idp-provider); every
# downstream notebook + script reads it back. (The Okta path also needs
# OKTA_ORG_URL + OKTA_API_TOKEN in .env.)
IDP_PROVIDER = set_idp_provider(ssm_client, value="cognito")  # ← change to "okta" for the Okta path
print(f"✅ IDP_PROVIDER persisted to SSM (/app/lakehouse-agent/idp-provider): {IDP_PROVIDER}")

## Step 1: Run IdP Setup

The two cells below are flag-guarded — **only the one matching `IDP_PROVIDER` runs**; the other prints a skip. (Both automated setups are retained; the flag selects which one executes.)

### [COGNITO] — user pool, OAuth clients, groups, test users

In [ ]:
import subprocess
import sys

if IDP_PROVIDER == "cognito":
    result = subprocess.run(
        [sys.executable, "setup_cognito.py"],
        cwd="deployment/1-cognito-setup",
        check=True,
    )
    print("\n✅ Cognito setup complete — configuration saved to SSM Parameter Store")
else:
    print(f"⏭️  Skipping Cognito setup (IDP_PROVIDER='{IDP_PROVIDER}').")

### [OKTA] — OIDC login app, OBO exchange app, custom auth server, groups, test users

Reads `OKTA_ORG_URL` + `OKTA_API_TOKEN` from your `.env`. Creates two Okta apps (login + a dedicated OBO token-exchange service app), the custom authorization server, scopes, groups, and test users.

#### `[OKTA]` Pre-flight: validate `.env` before deploying

Checks `OKTA_ORG_URL` and `OKTA_API_TOKEN` and **raises with the exact fix** if
either is missing or malformed — before `setup_okta.py` touches Okta at all.

It also loads `.env` into this kernel so the setup subprocess inherits it:
`setup_okta.py` reads `os.environ` directly, and nothing else in this notebook
loads `.env`.

Two malformations are easy to make and obscure to debug:

- **A scheme in the org URL.** The value is persisted to SSM *bare* and every
  consumer prepends `https://` itself, so a stored scheme becomes
  `https://https://dev-…` and raises `MissingSchema`.
- **The `-admin` hostname.** `dev-12345678-admin.okta.com` is the admin console
  you were just looking at to mint the API token, but the Okta management API is
  not served there — calls fail in a way that reads like a permissions problem.

In [ ]:
# [OKTA] Pre-flight: fail fast on unusable Okta configuration.
#
# Runs BEFORE setup_okta.py so a typo costs seconds instead of failing partway
# through a multi-hour deploy. It also loads .env into this kernel's environment,
# which the subprocess in the next cell inherits -- setup_okta.py reads
# os.environ directly and nothing else in this notebook loads .env.
#
# Two traps this catches that are otherwise obscure:
#   1. A scheme in OKTA_ORG_URL. The value is persisted to SSM *bare*; every
#      consumer prepends https:// itself, so a stored scheme becomes
#      "https://https://..." and requests raises MissingSchema.
#   2. The -admin hostname. dev-12345678-admin.okta.com is the admin console you
#      minted the API token in, but the Okta management API is not served there,
#      so calls fail in ways that read like a permissions error.
if IDP_PROVIDER == "okta":
    import os

    from utils.env_file import load_env_file

    # Shared loader (utils/env_file.py), the same one setup_okta.py calls, so the
    # notebook and the script path resolve the SAME .env. Already-exported
    # variables win, matching dotenv's default.
    #
    # A cwd-relative lookup was wrong here: this notebook runs at the sample root
    # while .env commonly lives below it in deployment/, and an upward-only search
    # walks toward the filesystem root, so it can never reach a subdirectory. The
    # helper searches upward first, then falls back to a bounded set of
    # sample-root-relative locations. It prints the path it used -- worth reading,
    # because .env.example ships at the sample root while an operator working in
    # the deployment tree may well have created deployment/.env instead.
    resolved_env_path, _loaded_names = load_env_file()
    if resolved_env_path is None:
        print("No .env found -- relying on the exported environment")

    problems = []
    org_url = (os.environ.get("OKTA_ORG_URL") or "").strip()
    api_token = (os.environ.get("OKTA_API_TOKEN") or "").strip()

    if not org_url:
        problems.append(
            "OKTA_ORG_URL not set. Add the bare tenant host to .env -- no scheme, "
            "no -admin suffix: OKTA_ORG_URL=dev-12345678.okta.com"
        )
    else:
        bare = org_url.split("://", 1)[-1].strip("/")
        if "://" in org_url:
            problems.append(
                f"OKTA_ORG_URL carries a scheme ({org_url}). Store it bare: "
                f"OKTA_ORG_URL={bare} -- consumers prepend https:// themselves, so a "
                "stored scheme yields https://https://... (requests MissingSchema)."
            )
        if "-admin." in bare:
            fixed = bare.replace("-admin.", ".", 1)
            problems.append(
                f"OKTA_ORG_URL points at the admin console ({bare}). The management "
                f"API is not served there. Drop the '-admin' segment: "
                f"OKTA_ORG_URL={fixed}"
            )
        if "/" in bare:
            problems.append(
                f"OKTA_ORG_URL must be a host with no path ({bare}). "
                f"Use OKTA_ORG_URL={bare.split('/', 1)[0]}"
            )

    if not api_token:
        problems.append(
            "OKTA_API_TOKEN not set. Mint one in the Okta admin console under "
            "Security -> API -> Tokens, then add OKTA_API_TOKEN=... to .env"
        )

    if problems:
        raise RuntimeError(
            "Okta configuration is unusable. Fix these in .env, then re-run this cell:\n"
            + "\n".join(f"  {n}. {p}" for n, p in enumerate(problems, 1))
        )

    print(f"OK   OKTA_ORG_URL   = {org_url}")
    print(f"OK   OKTA_API_TOKEN = set ({len(api_token)} chars; value not shown)")
    print("Pre-flight passed -- safe to run setup_okta.py in the next cell.")
else:
    print(f"Skipped: pre-flight is [OKTA]-only (IDP_PROVIDER='{IDP_PROVIDER}').")

In [ ]:
if IDP_PROVIDER == "okta":
    result = subprocess.run(
        [sys.executable, "setup_okta.py"],
        cwd="deployment/1-okta-setup",
        check=True,
    )
    print("\n✅ Okta setup complete — configuration saved to SSM Parameter Store")
else:
    print(f"⏭️  Skipping Okta setup (IDP_PROVIDER='{IDP_PROVIDER}').")

### [COGNITO] (Optional) — Login audit tracking

Cognito-only: deploys a Post-Authentication Lambda + DynamoDB table (`lakehouse_user_login_audit`) so administrators can query login history via the agent's `query_login_audit` tool. The Okta path has no equivalent — these cells skip when `IDP_PROVIDER != "cognito"`. **Skip entirely if you don't need login history.**

In [ ]:
import os

if IDP_PROVIDER == "cognito":
    # Ensure aws CLI is on PATH for the bash subprocess
    env = os.environ.copy()
    env["PATH"] = "/opt/homebrew/bin:/usr/local/bin:" + env.get("PATH", "")

    print("🚀 Deploying Login Audit Lambda and DynamoDB table...\n")

    result = subprocess.run(
        ["bash", "deploy_post_auth_lambda.sh"],
        cwd="deployment/1-cognito-setup",
        capture_output=True,
        text=True,
        env=env,
    )

    print(result.stdout)
    if result.returncode != 0:
        print("❌ Error:", result.stderr)
    else:
        print("\n✅ Login audit Lambda deployed!")
else:
    print(f"⏭️  Skipping login audit (Cognito-only; IDP_PROVIDER='{IDP_PROVIDER}').")

In [ ]:
if IDP_PROVIDER == "cognito":
    # Configure Cognito Post-Authentication trigger
    print("🔗 Configuring Cognito Post-Authentication trigger...\n")

    result = subprocess.run(
        [sys.executable, "setup_cognito.py", "--add-post-auth-trigger"],
        cwd="deployment/1-cognito-setup",
        capture_output=True,
        text=True,
    )

    print(result.stdout)
    if result.returncode != 0:
        print("❌ Error:", result.stderr)
    else:
        print("\n✅ Post-Authentication trigger configured!")
        print("   Login events will now be recorded to DynamoDB table: lakehouse_user_login_audit")
else:
    print(f"⏭️  Skipping post-auth trigger (Cognito-only; IDP_PROVIDER='{IDP_PROVIDER}').")

## Step 2: Verify Configuration in SSM

The setup scripts save all configuration to SSM Parameter Store. Run the cell matching your IdP to verify.

In [ ]:
# [COGNITO] Verify Cognito Configuration in SSM Parameter Store
if IDP_PROVIDER == "cognito":
    print("Verifying Cognito parameters in SSM Parameter Store...\n")

    # List of parameters to check
    parameters_to_check = [
        "/app/lakehouse-agent/cognito-user-pool-id",
        "/app/lakehouse-agent/cognito-app-client-id",
        "/app/lakehouse-agent/cognito-domain",
        "/app/lakehouse-agent/cognito-resource-server-id",
    ]

    # Check each parameter
    all_found = True
    for param_name in parameters_to_check:
        try:
            response = ssm_client.get_parameter(Name=param_name)
            value = response["Parameter"]["Value"]
            # Mask sensitive values
            display_value = value[:30] + "..." if len(value) > 30 else value
            print(f"✅ {param_name}")
            print(f"   Value: {display_value}")
        except ssm_client.exceptions.ParameterNotFound:
            print(f"❌ {param_name} - NOT FOUND")
            all_found = False
        except Exception as e:
            print(f"⚠️  {param_name} - ERROR: {e}")
            all_found = False

    # Check secure parameters (without displaying values)
    secure_params = [
        "/app/lakehouse-agent/cognito-app-client-secret",
    ]

    for param_name in secure_params:
        try:
            response = ssm_client.get_parameter(Name=param_name, WithDecryption=False)
            print(f"✅ {param_name} (SecureString)")
            print("   Value: ***MASKED***")
        except ssm_client.exceptions.ParameterNotFound:
            print(f"❌ {param_name} - NOT FOUND")
            all_found = False
        except Exception as e:
            print(f"⚠️  {param_name} - ERROR: {e}")

    if all_found:
        print("\n✅ All Cognito parameters verified in SSM Parameter Store!")
    else:
        print("\n⚠️  Some parameters are missing. Re-run the setup_cognito.py script.")
else:
    print(f"⏭️  Skipping Cognito SSM verify (IDP_PROVIDER='{IDP_PROVIDER}').")

In [ ]:
# [OKTA] Verify Okta configuration via the read-only verifier
if IDP_PROVIDER == "okta":
    result = subprocess.run(
        [sys.executable, "verify_okta_setup.py"],
        cwd="deployment/1-okta-setup",
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print("⚠️ ", result.stderr)
else:
    print(f"⏭️  Skipping Okta verify (IDP_PROVIDER='{IDP_PROVIDER}').")

### `[OKTA]` Step 2b: Live M2M token check

`verify_okta_setup.py` above is **config-only** — it reads SSM and inspects Okta
objects, but never requests a token. A broken scope or API access policy still
reads as healthy there. This cell mints a **real** token via `client_credentials`
against the custom authorization server, asking for the `claims.query` scope.

That is the first point where a broken auth-server policy actually surfaces.
Failing here costs seconds; discovering it at notebook 06 costs a redeploy.

It uses the **user-login app's** `client_id`/`secret` (SSM `okta-app-client-id` /
`okta-app-client-secret`) plus `okta-auth-server-id` — all written by
`setup_okta.py` in Step 1, so this must run after it.

In [ ]:
# [OKTA] Live M2M token check -- proves the Okta config can actually mint a token.
#
# Uses the user-login app's client_id/secret plus the custom auth-server id, all
# read from SSM (written by setup_okta.py in Step 1). The client-credentials grant
# and the claims.query scope are both enabled by the auth-server access policy
# that setup_okta.py creates, so this succeeds on a clean run.
if IDP_PROVIDER == "okta":
    result = subprocess.run(
        [sys.executable, "check_m2m_client.py"],
        cwd="deployment/1-okta-setup",
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(
            "Okta M2M token check FAILED -- see the output above. Most common causes: "
            "(1) a stale client secret, fixed by re-running setup_okta.py; or "
            "(2) the claims.query scope not granted by the auth-server access policy. "
            "Fix this before continuing -- later notebooks depend on this token path."
        )
else:
    print(f"Skipped: M2M check is [OKTA]-only (IDP_PROVIDER='{IDP_PROVIDER}').")

## Summary

✅ **Identity Provider Deployment Complete!** — Deployed IdP: **`{IDP_PROVIDER}`** (see Step 0).

**Test users (both providers create the same logical set):**
- policyholder001@example.com → policyholders group
- policyholder002@example.com → policyholders group
- adjuster001@example.com → adjusters group
- adjuster002@example.com → adjusters group
- admin@example.com → administrators group

_Cognito seeds these with password `TempPass123!` (first-login password-change challenge handled by the Streamlit UI). Okta seeds them in the Okta tenant._

**Next Steps:** Run **02-deploy-iam-roles.ipynb** (identity-provider-agnostic).